# DS605 Challenge: Black-Box Hyperparameter Optimization (from scratch)

**Idea.** Every Oracle call is expensive, so I fit a cheap *surrogate model* of the loss and let it decide where to look next. Everything below is hand-written (only `numpy` for linear algebra; no HPO/search library).

1. **Encode** each allowed value as a number: numeric grids -> rank in [0,1] (works for log-spaced grids), `None` -> top of the scale + flag, strings/bools -> one-hot.
2. **Initial design (~4-6 calls):** the middle of every grid (a sensible default) + max-min-distance points so the space is covered.
3. **Surrogate:** Gaussian Process (Matern-5/2 kernel) written with Cholesky; length-scales per parameter (ARD) and noise chosen by maximizing marginal likelihood. Losses are standardized (log-transformed if they span >10x).
4. **Acquisition:** Expected Improvement, evaluated on every unseen configuration (or 30k random + neighbours of the best ones if the grid is huge). The argmax is the next Oracle call.
5. **Never repeat a config** (repeats are counted) and **cache every result to `oracle_log.json`**, so re-running the notebook does not spend extra calls.
6. **Stop early** when expected improvement is ~0, or the best has stalled for `patience` calls with little expected upside; hard cap = `BUDGET`.

The API key is read from an environment variable / the cell below; **remove it before pushing to GitHub**.

In [ ]:
import json, os
from urllib.request import Request, urlopen

TEAM_ID = os.environ.get("TEAM_ID", "YOUR_TEAM_ID")
API_KEY = os.environ.get("API_KEY", "YOUR_API_KEY")   # <- remove before making the repo public
API_URL = os.environ.get("API_URL", "YOUR_API_URL")

def api(payload):
    req = Request(API_URL, data=json.dumps(payload).encode(),
                  headers={"Content-Type": "application/json", "X-API-Key": API_KEY})
    with urlopen(req, timeout=30) as r:
        return json.load(r)

# ---- Task 1: assigned search space (does not query the model)
spec = api({"action": "spec", "team_id": TEAM_ID})
SPACE = spec["hyperparameters"]
print("Model:", spec["model"])
for name, values in SPACE.items():
    print(name, ":", values)

# ---- Task 2: the only function that touches the Oracle
def oracle_query(params):
    result = api({"action": "query", "team_id": TEAM_ID, "params": params})
    return float(result["loss"])

## Optimizer (written from scratch)

In [ ]:
import itertools, json, math, os, random
import numpy as np

# ---------------------------------------------------------------- encoding
def build_encoder(space):
    """Map every allowed value of every parameter to a small float vector.
    Numeric params -> 1 column (rank in sorted order, scaled to [0,1]; rank
    handles log-spaced grids). Anything else (str/None/bool) -> one-hot."""
    names = list(space)
    table, owner = {}, []            # owner[j] = which param column j belongs to
    for p, n in enumerate(names):
        vals = space[n]
        isnum = lambda v: isinstance(v, (int, float)) and not isinstance(v, bool)
        nums = [i for i, v in enumerate(vals) if isnum(v)]
        nones = [i for i, v in enumerate(vals) if v is None]
        if nums and len(nums) + len(nones) == len(vals):
            # numeric (optionally with None, e.g. max_depth=None = "unbounded" -> top of the scale)
            order = sorted(nums, key=lambda i: vals[i])
            k = max(len(order) - 1 + (1 if nones else 0), 1)
            rank = {i: r / k for r, i in enumerate(order)}
            for i in nones: rank[i] = 1.0
            if nones:
                table[n] = [[rank[i], 1.0 if vals[i] is None else 0.0] for i in range(len(vals))]
                owner.extend([p, p])
            else:
                table[n] = [[rank[i]] for i in range(len(vals))]
                owner.append(p)
        else:                                          # strings / bools / mixed -> one-hot
            k = len(vals)
            table[n] = [[1.0 if j == i else 0.0 for j in range(k)] for i in range(k)]
            owner.extend([p] * k)
    return names, table, np.array(owner)

def encode(cfgs, names, table):
    return np.array([sum((table[n][i] for n, i in zip(names, c)), []) for c in cfgs], float)

# ---------------------------------------------------------------- GP surrogate
def matern52(A, B, inv_ls):
    A, B = A * inv_ls, B * inv_ls
    d2 = (A**2).sum(1)[:, None] + (B**2).sum(1)[None, :] - 2 * A @ B.T
    r = np.sqrt(np.maximum(d2, 0)) * math.sqrt(5)
    return (1 + r + r**2 / 3) * np.exp(-r)

def gp_fit(X, y, inv_ls, noise):
    K = matern52(X, X, inv_ls) + (noise + 1e-8) * np.eye(len(X))
    try:
        L = np.linalg.cholesky(K)
    except np.linalg.LinAlgError:
        return None
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y))
    lml = -0.5 * y @ alpha - np.log(np.diag(L)).sum()
    return L, alpha, lml

def fit_hyper(X, y, owner, n_params):
    """Pick length-scales + noise by maximising marginal likelihood
    (isotropic grid search, then per-parameter coordinate search = ARD)."""
    def score(ls, nz):
        r = gp_fit(X, y, 1.0 / ls[owner], nz)
        return -np.inf if r is None else r[2]
    best = (-np.inf, None, None)
    for l in (0.15, 0.3, 0.5, 0.8, 1.2, 2.0):
        for nz in (1e-6, 1e-3, 1e-2, 1e-1):
            s = score(np.full(n_params, l), nz)
            if s > best[0]:
                best = (s, np.full(n_params, l), nz)
    s, ls, nz = best
    if len(X) >= 2 * n_params:                       # enough data to justify ARD
        for _ in range(2):
            for p in range(n_params):
                for m in (0.5, 2.0):
                    t = ls.copy(); t[p] = np.clip(t[p] * m, 0.05, 5.0)
                    st = score(t, nz)
                    if st > s: s, ls = st, t
    return ls, nz

def gp_predict(Xs, X, L, alpha, inv_ls):
    Ks = matern52(Xs, X, inv_ls)
    mu = Ks @ alpha
    v = np.linalg.solve(L, Ks.T)
    var = np.maximum(1.0 - (v**2).sum(0), 1e-12)
    return mu, np.sqrt(var)

_erf = np.vectorize(math.erf)
def expected_improvement(mu, sd, best):
    z = (best - mu) / sd
    cdf = 0.5 * (1 + _erf(z / math.sqrt(2)))
    pdf = np.exp(-0.5 * z**2) / math.sqrt(2 * math.pi)
    return (best - mu) * cdf + sd * pdf

# ---------------------------------------------------------------- candidates
def all_candidates(space, rng, limit=30000, extra_around=()):
    sizes = [len(space[n]) for n in space]
    total = math.prod(sizes)
    if total <= limit:
        return list(itertools.product(*[range(s) for s in sizes]))
    cands = {tuple(rng.randrange(s) for s in sizes) for _ in range(limit)}
    for c in extra_around:                            # all 1-step neighbours of the best points
        for j, s in enumerate(sizes):
            for i in range(s):
                n = list(c); n[j] = i; cands.add(tuple(n))
    return list(cands)

# ---------------------------------------------------------------- main loop
def optimize(oracle, space, budget=30, n_init=None, patience=10, ei_tol=1e-2, ei_stall=2e-2,
             log_file="oracle_log.json", seed=0, verbose=True):
    rng = random.Random(seed)
    names, table, owner = build_encoder(space)
    n_params = len(names)
    sizes = [len(space[n]) for n in names]

    # history is persisted, so re-running the notebook never re-spends oracle calls
    hist = []                                          # list of (cfg_index_tuple, loss)
    if log_file and os.path.exists(log_file):
        for rec in json.load(open(log_file)):
            hist.append((tuple(space[n].index(rec["params"][n]) for n in names), rec["loss"]))
    seen = {c for c, _ in hist}

    def query(cfg):
        params = {n: space[n][i] for n, i in zip(names, cfg)}
        loss = float(oracle(params))
        hist.append((cfg, loss)); seen.add(cfg)
        if log_file:
            json.dump([{"params": {n: space[n][i] for n, i in zip(names, c)}, "loss": l}
                       for c, l in hist], open(log_file, "w"), indent=1)
        if verbose:
            print(f"call {len(hist):>3}  loss={loss:.6g}  best={min(l for _, l in hist):.6g}  {params}")
        return loss

    # ---- 1) small space-filling initial design (centre point + max-min picks)
    n_init = n_init or min(6, max(4, n_params))
    pool = all_candidates(space, rng, limit=3000)
    Xp = encode(pool, names, table)
    if not hist:
        query(tuple(s // 2 for s in sizes))            # middle of every grid = sensible default guess
    while len(hist) < n_init and len(hist) < budget:
        Xh = encode([c for c, _ in hist], names, table)
        d = np.min(((Xp[:, None, :] - Xh[None, :, :])**2).sum(2), axis=1)
        d[[i for i, c in enumerate(pool) if c in seen]] = -1
        query(pool[int(np.argmax(d))])

    # ---- 2) Bayesian loop: GP surrogate + Expected Improvement
    since_best, low_ei = 0, 0
    best_so_far = min(l for _, l in hist)
    while len(hist) < budget:
        cfgs = [c for c, _ in hist]
        y = np.array([l for _, l in hist])
        use_log = (y > 0).all() and y.max() / y.min() > 10
        yt = np.log(y) if use_log else y.copy()
        mu0, sd0 = yt.mean(), yt.std() or 1.0
        z = (yt - mu0) / sd0
        X = encode(cfgs, names, table)
        ls, nz = fit_hyper(X, z, owner, n_params)
        inv_ls = 1.0 / ls[owner]
        L, alpha, _ = gp_fit(X, z, inv_ls, nz)

        top = [c for c, _ in sorted(hist, key=lambda t: t[1])[:3]]
        cands = [c for c in all_candidates(space, rng, extra_around=top) if c not in seen]
        if not cands:
            break
        mu, sd = gp_predict(encode(cands, names, table), X, L, alpha, inv_ls)
        ei = expected_improvement(mu, sd, z.min())
        j = int(np.argmax(ei))

        # stopping rule: expected gain is negligible (in units of loss std) several times in a row
        low_ei = low_ei + 1 if ei[j] < ei_tol else 0
        if low_ei >= 2:
            if verbose: print("stop: expected improvement ~ 0")
            break
        query(cands[j])
        cur = min(l for _, l in hist)
        since_best = 0 if cur < best_so_far - 1e-12 else since_best + 1
        best_so_far = min(best_so_far, cur)
        if since_best >= patience and ei[j] < ei_stall:   # stalled AND model sees little upside
            if verbose: print(f"stop: no improvement in {patience} calls and low expected gain")
            break

    bc, bl = min(hist, key=lambda t: t[1])
    return {n: space[n][i] for n, i in zip(names, bc)}, bl, hist


## Run

In [ ]:
BUDGET = 30        # hard cap on Oracle calls; lower it if you want fewer calls, raise it if the curve is still dropping
best_params, best_loss, history = optimize(oracle_query, SPACE, budget=BUDGET, log_file="oracle_log.json", seed=0)

## Results (queried configs, best params, best loss, number of calls)

In [ ]:
print("=" * 60)
print("Best hyperparameters:", best_params)
print("Best loss           :", best_loss)
print("Total Oracle calls  :", len(history))

# every queried configuration and its loss, in query order
names = list(SPACE)
print("\n#  loss        " + "  ".join(names))
for k, (cfg, loss) in enumerate(history, 1):
    print(f"{k:<3}{loss:<12.6g}" + "  ".join(str(SPACE[n][i]) for n, i in zip(names, cfg)))

# convergence: best-so-far after each call
run_best, cur = [], float("inf")
for _, l in history:
    cur = min(cur, l); run_best.append(cur)
print("\nBest-so-far:", [round(v, 5) for v in run_best])